# 🧠 Psychology Chatbot

**How to run:** In the top menu, click **Runtime → Run all**. Along the way it will ask you for two things:
1. Your **Psychology.pdf** file (click "Choose Files")
2. Your **Groq API key** (paste it into the box)

At the very end you get a **public link** (`https://....gradio.live`). Click it and the chatbot opens.

## Step 1: Install the required libraries (1-2 minutes)

In [ ]:
!pip install -q langchain-community langchain-text-splitters langchain-huggingface langchain-groq faiss-cpu sentence-transformers pypdf gradio

## Step 2: Upload Psychology.pdf

In [ ]:
from google.colab import files

uploaded = files.upload()          # Click "Choose Files" and select Psychology.pdf
pdf_name = list(uploaded.keys())[0]
print("Uploaded:", pdf_name)

## Step 3: Enter your Groq API key
How to get a key (free): go to https://console.groq.com → log in with Google → **API Keys** → **Create API Key** → copy it.
A box will appear below. Paste the key and press Enter.

In [ ]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Paste your Groq API key and press Enter: ")
print("Key saved ✅")

## Step 4: Read the book and split it into small pieces

In [ ]:
import re
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Remove repeated header/footer lines from every page
NOISE = re.compile(r"^(Saylor URL:.*|Saylor\.org|\d{1,4})$", re.IGNORECASE)

pages = PyPDFLoader(pdf_name).load()
clean_pages = []
for p in pages:
    lines = [l for l in p.page_content.splitlines() if l.strip() and not NOISE.match(l.strip())]
    p.page_content = "\n".join(lines)
    p.metadata["page"] = int(p.metadata.get("page", 0)) + 1   # page numbers start at 1
    if len(p.page_content.split()) >= 30:                      # skip empty pages
        clean_pages.append(p)

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
chunks = splitter.split_documents(clean_pages)
print(f"Read {len(pages)} pages -> created {len(chunks)} chunks")

## Step 5: Make the book searchable (takes 3-6 minutes, please wait)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# This model understands both English and Hindi, so questions in either language work
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
db = FAISS.from_documents(chunks, embeddings)
retriever = db.as_retriever(search_kwargs={"k": 6})
print("Book is ready ✅")

## Step 6: Start the chatbot

In [ ]:
import gradio as gr
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

SYSTEM = (
    "You are a friendly psychology tutor. Answer the student's question using ONLY the "
    "textbook context below. Mention the page for key points like (p. 317). If the context "
    "does not contain the answer, say that the textbook doesn't seem to cover this, and do not "
    "make anything up. Keep the answer clear and simple (about 3-6 sentences). "
    "Reply in the same language the student used. You are an educational tool, not a "
    "therapist: for personal mental-health worries, share the textbook's general information "
    "and suggest talking to a qualified professional."
    "\n\nContext:\n{context}"
)
prompt = ChatPromptTemplate.from_messages([("system", SYSTEM), ("human", "{question}")])
chain = prompt | llm


def respond(message, history):
    docs = retriever.invoke(message)
    context = "\n\n".join(f"[p. {d.metadata['page']}] {d.page_content}" for d in docs)
    return chain.invoke({"context": context, "question": message}).content


# ---------- Look and feel ----------
theme = gr.themes.Soft(
    primary_hue="indigo",
    secondary_hue="purple",
    neutral_hue="slate",
    radius_size="lg",
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "sans-serif"],
)

css = """
.gradio-container { max-width: 880px !important; margin: auto !important; }
h1 {
    text-align: center;
    font-size: 2.6rem !important;
    font-weight: 800 !important;
    background: linear-gradient(90deg, #6366f1, #8b5cf6, #ec4899);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    margin-top: 12px !important;
}
.message, .bubble-wrap .message-row .message { border-radius: 16px !important; line-height: 1.6; }
footer { display: none !important; }
"""


def build(**extra):
    return gr.ChatInterface(
        fn=respond,
        title="Psychology Chatbot",
        chatbot=gr.Chatbot(
            height=520,
            show_label=False,
            placeholder="<h3>🧠 Hi! Ask me anything about psychology</h3>"
                        "<p>Pick an example below or type your own question.</p>",
        ),
        textbox=gr.Textbox(placeholder="Ask a psychology question...", container=False, scale=7),
        examples=[
            "What is classical conditioning?",
            "How does the fight-or-flight response work?",
            "What did Piaget say about child development?",
            "What is the difference between short-term and long-term memory?",
        ],
        **extra,
    )


# Works with both older and newer Gradio versions
try:
    demo = build(theme=theme, css=css)
    launch_extra = {}
except TypeError:
    demo = build()
    launch_extra = {"theme": theme, "css": css}

demo.launch(share=True, debug=True, **launch_extra)    # Click the gradio.live link that appears below